# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(url)

# Print basic dataset metadata
meta = dataset.metadata
print(f"Dataset: {meta.name}")
print(f"Description: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. All references to data entities use the `@id` for clarity and reproducibility.


In [ ]:
# List all record sets and their fields by @id
print('Record Sets and Fields:')
record_sets = dataset.record_sets

for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}, name: {getattr(rs, 'name', None)}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, name: {getattr(field, 'name', None)}, column @id: {getattr(field, 'column', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.


In [ ]:
# Extract data from all record sets into DataFrames (by @id)
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]
print(f"Record set @id list: {record_set_ids}")

for rs_id in record_set_ids:
    # List of dicts for this record set
    try:
        recs = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from {rs_id}")
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# Display the columns of the main record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Available columns in DataFrame for {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. All field and column references are by `@id`.


In [ ]:
# For this dataset, let's pick a numeric field (e.g., 'MW_kDa' - molecular weight of protein, by its column @id).
# Note: The column id must match the @id in the Croissant schema.
# Replace with the dataset's real @id for the MW field. Example: 'cr:MW_kDa'

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Try common possibilities for field ids (inspect DataFrame columns above to set the right @id)
    print(f"Columns available: {df.columns.tolist()}")
    # Let's try 'cr:MW_kDa' if present
    numeric_field_id = None
    common_num_fields = ['cr:MW_kDa', 'cr:MW', 'cr:coverage(%)', 'cr:peptide_count', 'cr:abundance_s1']
    for field in common_num_fields:
        if field in df.columns:
            numeric_field_id = field
            break
    if numeric_field_id is None:
        numeric_field_id = df.columns[df.dtypes == 'float64'][0] if any(df.dtypes == 'float64') else df.columns[0]
    print(f"Chosen numeric field for EDA: {numeric_field_id}")

    # Filter for proteins with MW > 10 kDa
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id}: (mean=0, std=1)")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a protein type, if available as a field (e.g. 'cr:protein_type', or first string column)
    group_field_id = None
    for col in df.columns:
        if col.startswith('cr:') and df[col].dtype == object and col != numeric_field_id:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No categorical group field found for grouping in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot a histogram for the selected numeric field, and a boxplot by group (if available).


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id in df:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if possible
    if group_field_id:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
    else:
        print("No suitable grouping field found for boxplot.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. 

- We loaded metadata and listed all available record sets and their fields using `@id`s.
- We extracted and inspected data from each record set and performed numeric analysis on a key quantitative field.
- We visualized the distribution and, if available, the differences between protein groups.

Users can adapt this notebook to further analyze subsets of the data or integrate it into downstream ML or bioinformatics workflows.